# 🎙️ AIC2026 — TASK-ING-02b: ASR Voice Transcript Extraction (Kaggle Notebook)

Notebook này phục vụ chạy **TASK-ING-02b**: Trích xuất giọng nói lời thoại từ video bằng **PhoWhisper (Tiếng Việt)** trên Kaggle GPU (T4/P100).

### 📌 ĐẦU VÀO:
- `/kaggle/working/shard_output/scene_metadata.parquet` (từ TASK-ING-01)
- `/kaggle/input/aic2026-videos/` (Thư mục chứa video gốc)

### 📌 ĐẦU RA:
- `/kaggle/working/shard_output/asr_text.parquet` `(scene_id, video_id, asr_text)`

In [ ]:
# ========================================== #
# CELL 1: Cài đặt các thư viện phụ thuộc    #
# ========================================== #
!pip install -q transformers soundfile accelerate pyyaml pandas pyarrow

In [ ]:
# ========================================== #
# CELL 2: Clone/Pull Mã nguồn Hệ thống       #
# ========================================== #
import os
if not os.path.exists('Super-Image-Retrieval-System-for-AIC2026'):
    !git clone https://github.com/ThanhDanh1510/Super-Image-Retrieval-System-for-AIC2026.git
%cd Super-Image-Retrieval-System-for-AIC2026
!git pull origin main

In [ ]:
# ================================================================= #
# CELL 3: Kiểm tra GPU & FFmpeg Audio Processor                     #
# ================================================================= #
import torch
import subprocess
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

# Kiểm tra ffmpeg
res = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
print(f"FFmpeg Status: {res.stdout.splitlines()[0] if res.returncode==0 else 'Not Found'}")

In [ ]:
# ================================================================= #
# CELL 4: Chạy ASR PhoWhisper trích xuất giọng nói Tiếng Việt       #
# ================================================================= #
!python -m ingestion.skeletons.01b_text_extraction \
    --metadata-path /kaggle/working/shard_output/scene_metadata.parquet \
    --keyframe-dir /kaggle/working/shard_output/keyframes \
    --video-dir /kaggle/input/aic2026-videos \
    --output-dir /kaggle/working/shard_output \
    --skip-ocr

In [ ]:
# ========================================== #
# CELL 5: Kiểm tra kết quả asr_text.parquet   #
# ========================================== #
import pandas as pd
asr_df = pd.read_parquet('/kaggle/working/shard_output/asr_text.parquet')
print(f"✅ Tổng số rows trong asr_text.parquet: {len(asr_df)}")
print("\n--- 10 ROWS MẪU --- ")
display(asr_df.head(10))

# Kiểm tra các phân cảnh có lời thoại
non_empty = asr_df[asr_df['asr_text'] != '']
print(f"\nSố scenes có nhận dạng được lời thoại ASR: {len(non_empty)}/{len(asr_df)}")